# DeepSeek Output Tokens

Compare the output lengths of DeepSeek's Regular Formalizer and Higher-Order Formalizer. The shared utilities in this notebook are domain-independent; each domain supplies its own output roots, artifact names, batches, and scaling variable.

The ChildSnack analysis counts the generated `problem.pddl` for the Regular Formalizer and the generated `generator.py` for the Higher-Order Formalizer. It uses the updated non-review outputs for `p01` through `p80`, averages token counts within each ten-problem batch, and plots the averages against the number of children.

The tokenizer is downloaded and cached at runtime. `deepseek-ai/DeepSeek-V3` is the default official published DeepSeek tokenizer; set `DEEPSEEK_TOKENIZER_ID` before running if a tokenizer specifically published for `deepseek-v4-flash` becomes available.

In [ ]:
# Cell 1: Install missing runtime dependencies into the active Jupyter kernel
import importlib.util
import subprocess
import sys

required_packages = {
    "matplotlib": "matplotlib",
    "transformers": "transformers",
}
missing_packages = [
    package_name
    for module_name, package_name in required_packages.items()
    if importlib.util.find_spec(module_name) is None
]

if missing_packages:
    print("Installing missing packages:", ", ".join(missing_packages))
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", *missing_packages]
    )
else:
    print("Required packages are already installed in this kernel.")

In [ ]:
# Cell 2: Imports, project paths, tokenizer configuration, and ChildSnack inputs
import os
from pathlib import Path
from statistics import fmean

import matplotlib.pyplot as plt
from transformers import AutoTokenizer


def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / "src").is_dir() and (candidate / "output").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate the formalizer-scaling project root.")


PROJECT_ROOT = find_project_root(Path.cwd())
TOKENIZER_ID = os.getenv("DEEPSEEK_TOKENIZER_ID", "deepseek-ai/DeepSeek-V3")
TOKENIZER_CACHE = PROJECT_ROOT / ".cache" / "huggingface"

CHILDSNACK_METHODS = {
    "Regular Formalizer": {
        "root": (
            PROJECT_ROOT
            / "output"
            / "direct_pddl_direct_unraveling"
            / "ChildSnack"
            / "single_sandwich_line"
            / "updated_deepseek_v4_flash"
        ),
        "artifact": "problem.pddl",
    },
    "Higher-Order Formalizer": {
        "root": (
            PROJECT_ROOT
            / "output"
            / "direct_pddl_unraveling"
            / "ChildSnack"
            / "single_sandwich_line"
            / "updated_deepseek_v4_flash"
        ),
        "artifact": "generator.py",
    },
}

CHILDSNACK_BATCHES = [
    {"start": 1, "end": 10, "size": 5},
    {"start": 11, "end": 20, "size": 10},
    {"start": 21, "end": 30, "size": 15},
    {"start": 31, "end": 40, "size": 20},
    {"start": 41, "end": 50, "size": 25},
    {"start": 51, "end": 60, "size": 30},
    {"start": 61, "end": 70, "size": 50},
    {"start": 71, "end": 80, "size": 100},
]

print("PROJECT_ROOT:", PROJECT_ROOT)
print("TOKENIZER_ID:", TOKENIZER_ID)
for method, config in CHILDSNACK_METHODS.items():
    print(f"{method}: {config['root']} -> {config['artifact']}")

In [ ]:
# Cell 3: Load the same DeepSeek tokenizer used by the reasoning-token notebook
TOKENIZER_CACHE.mkdir(parents=True, exist_ok=True)
tokenizer = AutoTokenizer.from_pretrained(
    TOKENIZER_ID,
    cache_dir=str(TOKENIZER_CACHE),
    use_fast=True,
)

print("Loaded tokenizer:", TOKENIZER_ID)
print("Tokenizer class:", tokenizer.__class__.__name__)
print("Cache directory:", TOKENIZER_CACHE)

In [ ]:
# Cell 4: Reusable output collection, token counting, and batch aggregation helpers
def problem_id(number: int) -> str:
    return f"p{number:02d}"


def collect_output_records(methods: dict, batches: list[dict]) -> list[dict]:
    records = []
    missing_files = []
    empty_files = []

    for method, config in methods.items():
        for batch in batches:
            batch_label = f"p{batch['start']:02d}-p{batch['end']:02d}"
            for number in range(batch["start"], batch["end"] + 1):
                pid = problem_id(number)
                path = config["root"] / pid / "cand_00" / config["artifact"]
                if not path.is_file():
                    missing_files.append(path)
                    continue

                text = path.read_text(encoding="utf-8")
                if not text.strip():
                    empty_files.append(path)
                    continue

                records.append(
                    {
                        "method": method,
                        "artifact": config["artifact"],
                        "batch": batch_label,
                        "problem_size": batch["size"],
                        "problem_id": pid,
                        "path": path,
                        "text": text,
                    }
                )

    if missing_files or empty_files:
        details = [
            *(f"Missing: {path}" for path in missing_files),
            *(f"Empty: {path}" for path in empty_files),
        ]
        raise FileNotFoundError(
            "Output coverage is incomplete; refusing to calculate biased averages.\n"
            + "\n".join(details)
        )

    expected_count = len(methods) * sum(
        batch["end"] - batch["start"] + 1 for batch in batches
    )
    if len(records) != expected_count:
        raise ValueError(f"Expected {expected_count} outputs; found {len(records)}.")

    return records


def count_output_tokens(text: str) -> int:
    return len(tokenizer.encode(text, add_special_tokens=False))


def calculate_batch_averages(
    records: list[dict], methods: dict, batches: list[dict]
) -> list[dict]:
    averages = []

    for batch in batches:
        batch_label = f"p{batch['start']:02d}-p{batch['end']:02d}"
        expected_batch_size = batch["end"] - batch["start"] + 1
        for method in methods:
            counts = [
                record["output_tokens"]
                for record in records
                if record["method"] == method and record["batch"] == batch_label
            ]
            if len(counts) != expected_batch_size:
                raise ValueError(
                    f"Expected {expected_batch_size} outputs for {method}, "
                    f"{batch_label}; found {len(counts)}."
                )

            averages.append(
                {
                    "method": method,
                    "batch": batch_label,
                    "problem_size": batch["size"],
                    "problem_count": len(counts),
                    "average_output_tokens": fmean(counts),
                }
            )

    return averages

## ChildSnack

In [ ]:
# Cell 5: Locate and validate all ChildSnack output artifacts
childsnack_records = collect_output_records(
    CHILDSNACK_METHODS, CHILDSNACK_BATCHES
)

outputs_per_method = len(childsnack_records) // len(CHILDSNACK_METHODS)
print(
    f"Validated {len(childsnack_records)} ChildSnack outputs "
    f"({outputs_per_method} per method)."
)

In [ ]:
# Cell 6: Count tokens in each ChildSnack output artifact
for record in childsnack_records:
    record["output_tokens"] = count_output_tokens(record["text"])

print("Per-problem output token counts:")
print(f"{'Method':<28} {'Artifact':<14} {'Problem':<8} {'Children':>8} {'Tokens':>10}")
print("-" * 74)
for record in childsnack_records:
    print(
        f"{record['method']:<28} "
        f"{record['artifact']:<14} "
        f"{record['problem_id']:<8} "
        f"{record['problem_size']:>8} "
        f"{record['output_tokens']:>10,}"
    )

In [ ]:
# Cell 7: Calculate average output tokens for each ChildSnack method and batch
childsnack_batch_averages = calculate_batch_averages(
    childsnack_records, CHILDSNACK_METHODS, CHILDSNACK_BATCHES
)

print("Average ChildSnack output tokens per batch:")
print(f"{'Method':<28} {'Batch':<11} {'Children':>8} {'Problems':>9} {'Average tokens':>16}")
print("-" * 78)
for row in childsnack_batch_averages:
    print(
        f"{row['method']:<28} "
        f"{row['batch']:<11} "
        f"{row['problem_size']:>8} "
        f"{row['problem_count']:>9} "
        f"{row['average_output_tokens']:>16,.2f}"
    )

In [ ]:
# Cell 8: Plot average ChildSnack output tokens against problem size
TITLE_FONTSIZE = 24
AXIS_LABEL_FONTSIZE = 20
TICK_LABEL_FONTSIZE = 17
LEGEND_FONTSIZE = 18
ANNOTATION_FONTSIZE = 15

from matplotlib.font_manager import FontProperties
from matplotlib.transforms import Bbox

styles = {
    "Regular Formalizer": {
        "color": "#0B6E75",
        "marker": "o",
        "label_positions": [
            (0, 18, "center", "bottom"),
            (-20, 14, "right", "bottom"),
            (20, 14, "left", "bottom"),
            (0, -20, "center", "top"),
        ],
    },
    "Higher-Order Formalizer": {
        "color": "#D97706",
        "marker": "s",
        "label_positions": [
            (0, -20, "center", "top"),
            (-20, -16, "right", "top"),
            (20, -16, "left", "top"),
            (0, 18, "center", "bottom"),
        ],
    },
}

LABEL_FALLBACK_POSITIONS = [
    (-22, -18, "right", "top"),
    (22, -18, "left", "top"),
    (0, 40, "center", "bottom"),
    (0, -42, "center", "top"),
    (38, 0, "left", "center"),
    (-38, 0, "right", "center"),
    (34, 28, "left", "bottom"),
    (-34, 28, "right", "bottom"),
    (34, -28, "left", "top"),
    (-34, -28, "right", "top"),
]
LABEL_FONT = FontProperties(size=ANNOTATION_FONTSIZE, weight="bold")


def _overlap_area(first, second):
    width = max(0, min(first.x1, second.x1) - max(first.x0, second.x0))
    height = max(0, min(first.y1, second.y1) - max(first.y0, second.y0))
    return width * height


def _label_bbox(anchor, width, height, horizontal_alignment, vertical_alignment, padding=4):
    anchor_x, anchor_y = anchor
    if horizontal_alignment == "center":
        left = anchor_x - width / 2
    elif horizontal_alignment == "right":
        left = anchor_x - width
    else:
        left = anchor_x

    if vertical_alignment == "center":
        bottom = anchor_y - height / 2
    elif vertical_alignment == "top":
        bottom = anchor_y - height
    else:
        bottom = anchor_y

    return Bbox.from_extents(
        left - padding,
        bottom - padding,
        left + width + padding,
        bottom + height + padding,
    )


def annotate_non_overlapping(ax, points, obstacle_artists=()):
    """Place crowded value labels first and choose the first collision-free position."""
    figure = ax.figure
    figure.canvas.draw()
    renderer = figure.canvas.get_renderer()
    axes_bbox = ax.get_window_extent(renderer=renderer)
    points_per_inch = figure.dpi / 72

    display_points = [ax.transData.transform((point["x"], point["y"])) for point in points]
    marker_boxes = [
        Bbox.from_extents(x - 8, y - 8, x + 8, y + 8)
        for x, y in display_points
    ]

    occupied = []
    for artist in obstacle_artists:
        if artist is not None:
            obstacle = artist.get_window_extent(renderer=renderer)
            occupied.append(
                Bbox.from_extents(
                    obstacle.x0 - 6, obstacle.y0 - 6, obstacle.x1 + 6, obstacle.y1 + 6
                )
            )

    crowded_first = []
    for index, display_point in enumerate(display_points):
        distances = [
            ((display_point[0] - other[0]) ** 2 + (display_point[1] - other[1]) ** 2) ** 0.5
            for other_index, other in enumerate(display_points)
            if other_index != index
        ]
        crowded_first.append((min(distances, default=float("inf")), index))

    for _, point_index in sorted(crowded_first):
        point = points[point_index]
        point_display = display_points[point_index]
        label = f"{point['y']:,.0f}"
        width, height, descent = renderer.get_text_width_height_descent(
            label, LABEL_FONT, ismath=False
        )
        height += descent

        preferred = point["style"]["label_positions"]
        candidates = preferred + [
            position for position in LABEL_FALLBACK_POSITIONS if position not in preferred
        ]
        best = None

        for priority, (offset_x, offset_y, horizontal_alignment, vertical_alignment) in enumerate(candidates):
            anchor = (
                point_display[0] + offset_x * points_per_inch,
                point_display[1] + offset_y * points_per_inch,
            )
            candidate_bbox = _label_bbox(
                anchor, width, height, horizontal_alignment, vertical_alignment
            )

            inside_width = max(
                0, min(candidate_bbox.x1, axes_bbox.x1) - max(candidate_bbox.x0, axes_bbox.x0)
            )
            inside_height = max(
                0, min(candidate_bbox.y1, axes_bbox.y1) - max(candidate_bbox.y0, axes_bbox.y0)
            )
            outside_area = candidate_bbox.width * candidate_bbox.height - inside_width * inside_height
            label_overlap = sum(_overlap_area(candidate_bbox, box) for box in occupied)
            marker_overlap = sum(
                _overlap_area(candidate_bbox, box)
                for marker_index, box in enumerate(marker_boxes)
                if marker_index != point_index
            )
            score = outside_area * 1_000_000 + label_overlap * 1_000 + marker_overlap * 10 + priority

            if best is None or score < best[0]:
                best = (
                    score, offset_x, offset_y, horizontal_alignment, vertical_alignment, candidate_bbox
                )
            if outside_area == 0 and label_overlap == 0 and marker_overlap == 0:
                break

        _, offset_x, offset_y, horizontal_alignment, vertical_alignment, label_bbox = best
        ax.annotate(
            label,
            xy=(point["x"], point["y"]),
            xytext=(offset_x, offset_y),
            textcoords="offset points",
            color=point["style"]["color"],
            fontsize=ANNOTATION_FONTSIZE,
            fontweight="bold",
            ha=horizontal_alignment,
            va=vertical_alignment,
        )
        occupied.append(label_bbox)

fig, ax = plt.subplots(figsize=(12.5, 7.5))
fig.patch.set_facecolor("#F6F1E8")
ax.set_facecolor("#FFFDF8")
childsnack_label_points = []

for method, style in styles.items():
    rows = sorted(
        (row for row in childsnack_batch_averages if row["method"] == method),
        key=lambda row: row["problem_size"],
    )
    x_values = [row["problem_size"] for row in rows]
    y_values = [row["average_output_tokens"] for row in rows]

    ax.plot(
        x_values,
        y_values,
        label=method,
        color=style["color"],
        marker=style["marker"],
        linewidth=3.2,
        markersize=10,
    )

    childsnack_label_points.extend(
        {"x": row["problem_size"], "y": row["average_output_tokens"], "style": style}
        for row in rows
    )

child_counts = [batch["size"] for batch in CHILDSNACK_BATCHES]
ax.set_xticks(child_counts)
ax.set_xlabel(
    "Number of children",
    fontsize=AXIS_LABEL_FONTSIZE,
    fontweight="semibold",
    labelpad=12,
)
ax.set_ylabel(
    "Average output tokens per problem",
    fontsize=AXIS_LABEL_FONTSIZE,
    fontweight="semibold",
    labelpad=12,
)
ax.set_title(
    "DeepSeek ChildSnack Output Tokens",
    fontsize=TITLE_FONTSIZE,
    fontweight="bold",
    pad=18,
)
ax.grid(axis="y", color="#D8D1C4", linewidth=0.8, alpha=0.8)
ax.spines[["top", "right"]].set_visible(False)
ax.tick_params(axis="both", labelsize=TICK_LABEL_FONTSIZE)
childsnack_legend = ax.legend(frameon=False, fontsize=LEGEND_FONTSIZE)
ax.ticklabel_format(axis="y", style="plain")
ax.margins(y=0.14)
fig.tight_layout()
annotate_non_overlapping(
    ax, childsnack_label_points, obstacle_artists=[childsnack_legend]
)
plt.show()

## Transport

The Transport analysis uses the current DeepSeek non-review outputs for `p01` through `p70`. These folders retain the `deepseek_v4_flash` name because Transport did not require replacement problem descriptions. The Regular Formalizer contributes `problem.pddl`, while the Higher-Order Formalizer contributes `generator.py`.

In [ ]:
# Cell 9: Transport output roots and location-count batches
TRANSPORT_METHODS = {
    "Regular Formalizer": {
        "root": (
            PROJECT_ROOT
            / "output"
            / "direct_pddl_direct_unraveling"
            / "Transport"
            / "fully_connected_delivery"
            / "deepseek_v4_flash"
        ),
        "artifact": "problem.pddl",
    },
    "Higher-Order Formalizer": {
        "root": (
            PROJECT_ROOT
            / "output"
            / "direct_pddl_unraveling"
            / "Transport"
            / "fully_connected_delivery"
            / "deepseek_v4_flash"
        ),
        "artifact": "generator.py",
    },
}

TRANSPORT_BATCHES = [
    {"start": 1, "end": 10, "size": 10},
    {"start": 11, "end": 20, "size": 15},
    {"start": 21, "end": 30, "size": 20},
    {"start": 31, "end": 40, "size": 25},
    {"start": 41, "end": 50, "size": 30},
    {"start": 51, "end": 60, "size": 50},
    {"start": 61, "end": 70, "size": 100},
]

for method, config in TRANSPORT_METHODS.items():
    print(f"{method}: {config['root']} -> {config['artifact']}")

In [ ]:
# Cell 10: Locate and validate all Transport output artifacts
transport_records = collect_output_records(TRANSPORT_METHODS, TRANSPORT_BATCHES)

outputs_per_method = len(transport_records) // len(TRANSPORT_METHODS)
print(
    f"Validated {len(transport_records)} Transport outputs "
    f"({outputs_per_method} per method)."
)

In [ ]:
# Cell 11: Count tokens in each Transport output artifact
for record in transport_records:
    record["output_tokens"] = count_output_tokens(record["text"])

print("Per-problem Transport output token counts:")
print(f"{'Method':<28} {'Artifact':<14} {'Problem':<8} {'Locations':>9} {'Tokens':>10}")
print("-" * 75)
for record in transport_records:
    print(
        f"{record['method']:<28} "
        f"{record['artifact']:<14} "
        f"{record['problem_id']:<8} "
        f"{record['problem_size']:>9} "
        f"{record['output_tokens']:>10,}"
    )

In [ ]:
# Cell 12: Calculate average output tokens for each Transport method and batch
transport_batch_averages = calculate_batch_averages(
    transport_records, TRANSPORT_METHODS, TRANSPORT_BATCHES
)

print("Average Transport output tokens per batch:")
print(f"{'Method':<28} {'Batch':<11} {'Locations':>9} {'Problems':>9} {'Average tokens':>16}")
print("-" * 79)
for row in transport_batch_averages:
    print(
        f"{row['method']:<28} "
        f"{row['batch']:<11} "
        f"{row['problem_size']:>9} "
        f"{row['problem_count']:>9} "
        f"{row['average_output_tokens']:>16,.2f}"
    )

In [ ]:
# Cell 13: Plot average Transport output tokens against problem size
transport_fig, transport_ax = plt.subplots(figsize=(12.5, 7.5))
transport_fig.patch.set_facecolor("#F6F1E8")
transport_ax.set_facecolor("#FFFDF8")
transport_label_points = []

for method, style in styles.items():
    rows = sorted(
        (row for row in transport_batch_averages if row["method"] == method),
        key=lambda row: row["problem_size"],
    )
    transport_ax.plot(
        [row["problem_size"] for row in rows],
        [row["average_output_tokens"] for row in rows],
        label=method,
        color=style["color"],
        marker=style["marker"],
        linewidth=3.2,
        markersize=10,
    )

    transport_label_points.extend(
        {"x": row["problem_size"], "y": row["average_output_tokens"], "style": style}
        for row in rows
    )

location_counts = [batch["size"] for batch in TRANSPORT_BATCHES]
transport_ax.set_xticks(location_counts)
transport_ax.set_xlabel(
    "Number of locations",
    fontsize=AXIS_LABEL_FONTSIZE,
    fontweight="semibold",
    labelpad=12,
)
transport_ax.set_ylabel(
    "Average output tokens per problem",
    fontsize=AXIS_LABEL_FONTSIZE,
    fontweight="semibold",
    labelpad=12,
)
transport_ax.set_title(
    "DeepSeek Transport Output Tokens",
    fontsize=TITLE_FONTSIZE,
    fontweight="bold",
    pad=18,
)
transport_ax.grid(axis="y", color="#D8D1C4", linewidth=0.8, alpha=0.8)
transport_ax.spines[["top", "right"]].set_visible(False)
transport_ax.tick_params(axis="both", labelsize=TICK_LABEL_FONTSIZE)
transport_legend = transport_ax.legend(frameon=False, fontsize=LEGEND_FONTSIZE)
transport_ax.ticklabel_format(axis="y", style="plain")
transport_ax.margins(y=0.14)
transport_fig.tight_layout()
annotate_non_overlapping(
    transport_ax, transport_label_points, obstacle_artists=[transport_legend]
)
plt.show()

## BlocksWorld

In [ ]:
# Cell 14: BlocksWorld output roots and block-count batches
BLOCKSWORLD_METHODS = {
    "Regular Formalizer": {
        "root": (
            PROJECT_ROOT
            / "output"
            / "direct_pddl_direct_unraveling"
            / "BlocksWorld"
            / "deepseek_v4_flash"
        ),
        "artifact": "problem.pddl",
    },
    "Higher-Order Formalizer": {
        "root": (
            PROJECT_ROOT
            / "output"
            / "direct_pddl_unraveling"
            / "BlocksWorld"
            / "deepseek_v4_flash"
        ),
        "artifact": "generator.py",
    },
}

BLOCKSWORLD_BATCHES = [
    {"start": 1, "end": 10, "size": 5},
    {"start": 51, "end": 60, "size": 30},
    {"start": 91, "end": 100, "size": 50},
    {"start": 141, "end": 150, "size": 75},
    {"start": 191, "end": 200, "size": 100},
]

for method, config in BLOCKSWORLD_METHODS.items():
    print(f"{method}: {config['root']} -> {config['artifact']}")

In [ ]:
# Cell 15: Locate and validate all BlocksWorld output artifacts
blocksworld_records = collect_output_records(BLOCKSWORLD_METHODS, BLOCKSWORLD_BATCHES)

outputs_per_method = len(blocksworld_records) // len(BLOCKSWORLD_METHODS)
print(
    f"Validated {len(blocksworld_records)} BlocksWorld outputs "
    f"({outputs_per_method} per method)."
)

In [ ]:
# Cell 16: Count tokens in each BlocksWorld output artifact
for record in blocksworld_records:
    record["output_tokens"] = count_output_tokens(record["text"])

print("Per-problem BlocksWorld output token counts:")
print(f"{'Method':<28} {'Artifact':<14} {'Problem':<8} {'Blocks':>7} {'Tokens':>10}")
print("-" * 73)
for record in blocksworld_records:
    print(
        f"{record['method']:<28} "
        f"{record['artifact']:<14} "
        f"{record['problem_id']:<8} "
        f"{record['problem_size']:>7} "
        f"{record['output_tokens']:>10,}"
    )

In [ ]:
# Cell 17: Calculate average output tokens for each BlocksWorld method and batch
blocksworld_batch_averages = calculate_batch_averages(
    blocksworld_records, BLOCKSWORLD_METHODS, BLOCKSWORLD_BATCHES
)

print("Average BlocksWorld output tokens per batch:")
print(f"{'Method':<28} {'Batch':<11} {'Blocks':>7} {'Problems':>9} {'Average tokens':>16}")
print("-" * 77)
for row in blocksworld_batch_averages:
    print(
        f"{row['method']:<28} "
        f"{row['batch']:<11} "
        f"{row['problem_size']:>7} "
        f"{row['problem_count']:>9} "
        f"{row['average_output_tokens']:>16,.2f}"
    )

In [ ]:
# Cell 18: Plot average BlocksWorld output tokens against problem size
blocksworld_fig, blocksworld_ax = plt.subplots(figsize=(12.5, 7.5))
blocksworld_fig.patch.set_facecolor("#F6F1E8")
blocksworld_ax.set_facecolor("#FFFDF8")
blocksworld_label_points = []

for method, style in styles.items():
    rows = sorted(
        (row for row in blocksworld_batch_averages if row["method"] == method),
        key=lambda row: row["problem_size"],
    )
    blocksworld_ax.plot(
        [row["problem_size"] for row in rows],
        [row["average_output_tokens"] for row in rows],
        label=method,
        color=style["color"],
        marker=style["marker"],
        linewidth=3.2,
        markersize=10,
    )

    blocksworld_label_points.extend(
        {"x": row["problem_size"], "y": row["average_output_tokens"], "style": style}
        for row in rows
    )

block_counts = [batch["size"] for batch in BLOCKSWORLD_BATCHES]
blocksworld_ax.set_xticks(block_counts)
blocksworld_ax.set_xlabel(
    "Number of blocks",
    fontsize=AXIS_LABEL_FONTSIZE,
    fontweight="semibold",
    labelpad=12,
)
blocksworld_ax.set_ylabel(
    "Average output tokens per problem",
    fontsize=AXIS_LABEL_FONTSIZE,
    fontweight="semibold",
    labelpad=12,
)
blocksworld_ax.set_title(
    "DeepSeek BlocksWorld Output Tokens",
    fontsize=TITLE_FONTSIZE,
    fontweight="bold",
    pad=18,
)
blocksworld_ax.grid(axis="y", color="#D8D1C4", linewidth=0.8, alpha=0.8)
blocksworld_ax.spines[["top", "right"]].set_visible(False)
blocksworld_ax.tick_params(axis="both", labelsize=TICK_LABEL_FONTSIZE)
blocksworld_legend = blocksworld_ax.legend(frameon=False, fontsize=LEGEND_FONTSIZE)
blocksworld_ax.ticklabel_format(axis="y", style="plain")
blocksworld_ax.margins(y=0.14)
blocksworld_fig.tight_layout()
annotate_non_overlapping(
    blocksworld_ax, blocksworld_label_points, obstacle_artists=[blocksworld_legend]
)
plt.show()

## Elevators

In [ ]:
# Cell 19: Elevators output roots and floor-count batches
ELEVATORS_METHODS = {
    "Regular Formalizer": {
        "root": (
            PROJECT_ROOT
            / "output"
            / "direct_pddl_direct_unraveling"
            / "Elevators"
            / "bottom_to_top_delivery"
            / "deepseek_v4_flash"
        ),
        "artifact": "problem.pddl",
    },
    "Higher-Order Formalizer": {
        "root": (
            PROJECT_ROOT
            / "output"
            / "direct_pddl_unraveling"
            / "Elevators"
            / "bottom_to_top_delivery"
            / "deepseek_v4_flash"
        ),
        "artifact": "generator.py",
    },
}

ELEVATORS_BATCHES = [
    {"start": 1, "end": 10, "size": 10},
    {"start": 11, "end": 20, "size": 15},
    {"start": 21, "end": 30, "size": 20},
    {"start": 31, "end": 40, "size": 25},
    {"start": 41, "end": 50, "size": 30},
    {"start": 51, "end": 60, "size": 50},
    {"start": 61, "end": 70, "size": 100},
]

for method, config in ELEVATORS_METHODS.items():
    print(f"{method}: {config['root']} -> {config['artifact']}")

In [ ]:
# Cell 20: Locate and validate all Elevators output artifacts
elevators_records = collect_output_records(ELEVATORS_METHODS, ELEVATORS_BATCHES)

outputs_per_method = len(elevators_records) // len(ELEVATORS_METHODS)
print(
    f"Validated {len(elevators_records)} Elevators outputs "
    f"({outputs_per_method} per method)."
)

In [ ]:
# Cell 21: Count tokens in each Elevators output artifact
for record in elevators_records:
    record["output_tokens"] = count_output_tokens(record["text"])

print("Per-problem Elevators output token counts:")
print(f"{'Method':<28} {'Artifact':<14} {'Problem':<8} {'Floors':>7} {'Tokens':>10}")
print("-" * 73)
for record in elevators_records:
    print(
        f"{record['method']:<28} "
        f"{record['artifact']:<14} "
        f"{record['problem_id']:<8} "
        f"{record['problem_size']:>7} "
        f"{record['output_tokens']:>10,}"
    )

In [ ]:
# Cell 22: Calculate average output tokens for each Elevators method and batch
elevators_batch_averages = calculate_batch_averages(
    elevators_records, ELEVATORS_METHODS, ELEVATORS_BATCHES
)

print("Average Elevators output tokens per batch:")
print(f"{'Method':<28} {'Batch':<11} {'Floors':>7} {'Problems':>9} {'Average tokens':>16}")
print("-" * 77)
for row in elevators_batch_averages:
    print(
        f"{row['method']:<28} "
        f"{row['batch']:<11} "
        f"{row['problem_size']:>7} "
        f"{row['problem_count']:>9} "
        f"{row['average_output_tokens']:>16,.2f}"
    )

In [ ]:
# Cell 23: Plot average Elevators output tokens against problem size
elevators_fig, elevators_ax = plt.subplots(figsize=(12.5, 7.5))
elevators_fig.patch.set_facecolor("#F6F1E8")
elevators_ax.set_facecolor("#FFFDF8")
elevators_label_points = []

for method, style in styles.items():
    rows = sorted(
        (row for row in elevators_batch_averages if row["method"] == method),
        key=lambda row: row["problem_size"],
    )
    elevators_ax.plot(
        [row["problem_size"] for row in rows],
        [row["average_output_tokens"] for row in rows],
        label=method,
        color=style["color"],
        marker=style["marker"],
        linewidth=3.2,
        markersize=10,
    )

    elevators_label_points.extend(
        {"x": row["problem_size"], "y": row["average_output_tokens"], "style": style}
        for row in rows
    )

floor_counts = [batch["size"] for batch in ELEVATORS_BATCHES]
elevators_ax.set_xticks(floor_counts)
elevators_ax.set_xlabel(
    "Number of floors",
    fontsize=AXIS_LABEL_FONTSIZE,
    fontweight="semibold",
    labelpad=12,
)
elevators_ax.set_ylabel(
    "Average output tokens per problem",
    fontsize=AXIS_LABEL_FONTSIZE,
    fontweight="semibold",
    labelpad=12,
)
elevators_ax.set_title(
    "DeepSeek Elevators Output Tokens",
    fontsize=TITLE_FONTSIZE,
    fontweight="bold",
    pad=18,
)
elevators_ax.grid(axis="y", color="#D8D1C4", linewidth=0.8, alpha=0.8)
elevators_ax.spines[["top", "right"]].set_visible(False)
elevators_ax.tick_params(axis="both", labelsize=TICK_LABEL_FONTSIZE)
elevators_legend = elevators_ax.legend(frameon=False, fontsize=LEGEND_FONTSIZE)
elevators_ax.ticklabel_format(axis="y", style="plain")
elevators_ax.margins(y=0.14)
elevators_fig.tight_layout()
annotate_non_overlapping(
    elevators_ax, elevators_label_points, obstacle_artists=[elevators_legend]
)
plt.show()